# ISIC 2018 skin-lesion segmentation: benchmark and training-protocol study

This notebook produces the experiments for the paper *Training Protocol Matters: A Controlled Study of U-Net Optimisation for Dermoscopic Skin Lesion Segmentation*.

**Experiment A: architecture benchmark.** Six models are trained with one shared recipe, 5 seeds each: U-Net (scratch), Attention U-Net (scratch), U-Net with a ResNet-34 encoder, UNet++, DeepLabV3+ and SegFormer (MiT-B2). The recipe is AdamW with a cosine schedule, BCE + Dice loss, augmentation, early stopping on validation Dice with best-weight restoration, and flip test-time augmentation.

**Experiment B: protocol replication.** The scratch U-Net is trained under the paper's four protocols T1–T4, with 5 seeds each, to measure run-to-run variance.

**Evaluation.**
- Test sets: the official ISIC 2018 Task 1 test set (1,000 images), plus external testing on ISIC 2017 (600 images) and, optionally, PH2.
- Metrics: accuracy, Dice, Jaccard, the ISIC thresholded Jaccard, precision, recall, specificity and HD95.
- Statistics: mean ± SD, 95% CI, Wilcoxon signed-rank tests with Holm correction, and a Friedman test.

**Outputs.** Paper-ready tables (Markdown and LaTeX), figures at 300 dpi, and a `summary.md`.

### How to run
1. **Runtime → Change runtime type → GPU** (a T4 is enough; an A100 or L4 is about 3× faster).
2. Check the settings in the first code cell, then **Runtime → Run all**.
3. If Colab disconnects, reconnect and **Run all** again. Finished runs are skipped and unfinished runs resume from their last checkpoint, because results are written to Google Drive.
4. When it finishes, download `results_bundle.zip` (last cell) and send it back for the paper.

### Compute budget (T4, 256 × 256)
| Part | Runs | Approx. GPU time |
|---|---|---|
| Data download + caching (once) | – | 20–40 min |
| Experiment A, 6 models × 5 seeds | 30 | 15–25 h |
| Experiment B, 4 protocols × 5 seeds | 20 | 5–8 h |

Free Colab sessions end after a few hours, so the run will span several sessions. Kaggle Notebooks (30 free GPU hours a week) and Colab Pro also work. The notebook detects Kaggle and writes to `/kaggle/working`. To get first results sooner, set `SEEDS = [0, 1, 2]`; you can add seeds 3 and 4 later.

In [ ]:
# ===================== SETTINGS =====================
IMG_SIZE      = 256                      # training/evaluation resolution
SEEDS         = [0, 1, 2, 3, 4]          # use [0, 1, 2] for a faster first pass
MODELS        = ["unet_scratch", "attunet_scratch", "unet_r34", "unetpp_r34", "deeplabv3p_r34", "segformer_b2"]
RUN_BENCHMARK = True                     # Experiment A
RUN_PROTOCOLS = True                     # Experiment B (T1-T4 replication)
EXTERNAL      = ["isic2017_test"]        # external test sets downloaded automatically
PH2_ROOT      = None                     # e.g. "/content/drive/MyDrive/PH2Dataset" after manual download, else None
RUN_SMOKE_TEST = True                    # 1-2 minute check of every model before the long runs
# ====================================================

In [ ]:
import os, sys, subprocess
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/isic2018_study"
    RAW = "/content/raw"                 # large zips stay on local disk, not Drive
elif IN_KAGGLE:
    PROJECT, RAW = "/kaggle/working/isic2018_study", "/kaggle/temp/raw"
else:
    PROJECT, RAW = os.path.abspath("isic2018_study"), os.path.abspath("raw")
CACHE, RESULTS, ANALYSIS = f"{PROJECT}/cache", f"{PROJECT}/results", f"{PROJECT}/analysis"
for d in (PROJECT, RAW, CACHE, RESULTS, ANALYSIS):
    os.makedirs(d, exist_ok=True)
CODE = f"{PROJECT}/code"; os.makedirs(CODE, exist_ok=True)
sys.path.insert(0, CODE)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch>=0.4.0", "tabulate"], check=True)
import torch, segmentation_models_pytorch as smp
print("torch", torch.__version__, "| smp", smp.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "No GPU - set Runtime > Change runtime type > GPU"

`isicseg_core.py`: written to the project folder so the analysis can be re-run later without the notebook.

In [ ]:
%%writefile {CODE}/isicseg_core.py
"""
isicseg_core.py - framework-free parts of the ISIC 2018 segmentation study.

Data download and caching, augmentation, per-image metrics (incl. HD95 and the
ISIC thresholded Jaccard), statistics across seeds, and paper figures.
Only numpy / scipy / pandas / opencv / matplotlib are needed here, so this file
can be tested without a GPU or PyTorch.
"""
from __future__ import annotations

import glob
import json
import math
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from scipy import ndimage, stats

# ----------------------------------------------------------------------------
# 1. Data
# ----------------------------------------------------------------------------
ISIC_BASE = "https://isic-challenge-data.s3.amazonaws.com"
ISIC_URLS = {
    "isic2018_train_img": f"{ISIC_BASE}/2018/ISIC2018_Task1-2_Training_Input.zip",
    "isic2018_train_gt": f"{ISIC_BASE}/2018/ISIC2018_Task1_Training_GroundTruth.zip",
    "isic2018_val_img": f"{ISIC_BASE}/2018/ISIC2018_Task1-2_Validation_Input.zip",
    "isic2018_val_gt": f"{ISIC_BASE}/2018/ISIC2018_Task1_Validation_GroundTruth.zip",
    "isic2018_test_img": f"{ISIC_BASE}/2018/ISIC2018_Task1-2_Test_Input.zip",
    "isic2018_test_gt": f"{ISIC_BASE}/2018/ISIC2018_Task1_Test_GroundTruth.zip",
    "isic2017_test_img": f"{ISIC_BASE}/2017/ISIC-2017_Test_v2_Data.zip",
    "isic2017_test_gt": f"{ISIC_BASE}/2017/ISIC-2017_Test_v2_Part1_GroundTruth.zip",
}
# which zip pairs make which dataset
DATASETS = {
    "isic2018_train": ("isic2018_train_img", "isic2018_train_gt"),
    "isic2018_val": ("isic2018_val_img", "isic2018_val_gt"),
    "isic2018_test": ("isic2018_test_img", "isic2018_test_gt"),
    "isic2017_test": ("isic2017_test_img", "isic2017_test_gt"),
}

MODEL_LABELS = {
    "unet_scratch": "U-Net (scratch)",
    "attunet_scratch": "Attention U-Net (scratch)",
    "unet_r34": "U-Net (ResNet-34)",
    "unetpp_r34": "UNet++ (ResNet-34)",
    "deeplabv3p_r34": "DeepLabV3+ (ResNet-34)",
    "segformer_b2": "SegFormer (MiT-B2)",
}

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def download(url: str, dest: Path) -> Path:
    """Download with wget (resumable). Skips if the file is already complete."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and zipfile.is_zipfile(dest):
        return dest
    print(f"Downloading {url}")
    subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", str(dest), url], check=True)
    if not zipfile.is_zipfile(dest):
        raise RuntimeError(f"{dest} is not a valid zip; delete it and retry")
    return dest


def extract(zip_path: Path, out_dir: Path) -> Path:
    out_dir = Path(out_dir)
    marker = out_dir / ".extracted"
    if marker.exists():
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {zip_path.name}")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)
    marker.touch()
    return out_dir


def pair_images_and_masks(img_dir: Path, mask_dir: Path):
    """Return sorted (image_id, image_path, mask_path) triples for ISIC folders."""
    masks = {}
    for p in glob.glob(str(Path(mask_dir) / "**" / "*_segmentation.png"), recursive=True):
        stem = Path(p).name.replace("_segmentation.png", "")
        masks[stem] = p
    pairs = []
    for p in glob.glob(str(Path(img_dir) / "**" / "*.jpg"), recursive=True):
        stem = Path(p).stem
        if "superpixels" in stem:
            continue
        if stem in masks:
            pairs.append((stem, p, masks[stem]))
    pairs.sort()
    return pairs


def pair_ph2(ph2_root: Path):
    """PH2 layout: <root>/IMDxxx/IMDxxx_Dermoscopic_Image/IMDxxx.bmp and IMDxxx_lesion/IMDxxx_lesion.bmp"""
    pairs = []
    for d in sorted(glob.glob(str(Path(ph2_root) / "**" / "IMD*_Dermoscopic_Image"), recursive=True)):
        case = Path(d).name.replace("_Dermoscopic_Image", "")
        img = Path(d) / f"{case}.bmp"
        msk = Path(d).parent / f"{case}_lesion" / f"{case}_lesion.bmp"
        if img.exists() and msk.exists():
            pairs.append((case, str(img), str(msk)))
    return pairs


def load_resize(pairs, size: int):
    """Read and resize. Images: INTER_AREA; masks: INTER_NEAREST, binarised at 127."""
    n = len(pairs)
    X = np.zeros((n, size, size, 3), np.uint8)
    Y = np.zeros((n, size, size), np.uint8)
    ids = []
    for i, (sid, ip, mp) in enumerate(pairs):
        img = cv2.imread(ip, cv2.IMREAD_COLOR)
        if img is None:
            raise IOError(ip)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        msk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if msk is None:
            raise IOError(mp)
        X[i] = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
        Y[i] = (cv2.resize(msk, (size, size), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8)
        ids.append(sid)
        if (i + 1) % 500 == 0:
            print(f"  {i + 1}/{n}")
    return X, Y, np.array(ids)


def build_cache(name: str, raw_dir: Path, cache_dir: Path, size: int, ph2_root: str | None = None) -> Path:
    """Download (if needed), resize and save one dataset as <cache>/<name>_<size>.npz."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    out = cache_dir / f"{name}_{size}.npz"
    if out.exists():
        return out
    if name == "ph2":
        if not ph2_root or not Path(ph2_root).exists():
            print("PH2 not found - skipped (download it manually, see notebook).")
            return None
        pairs = pair_ph2(Path(ph2_root))
    else:
        img_key, gt_key = DATASETS[name]
        dirs = []
        for key in (img_key, gt_key):
            z = download(ISIC_URLS[key], Path(raw_dir) / f"{key}.zip")
            dirs.append(extract(z, Path(raw_dir) / key))
        pairs = pair_images_and_masks(*dirs)
    if not pairs:
        raise RuntimeError(f"No image/mask pairs found for {name}")
    print(f"{name}: {len(pairs)} pairs -> resizing to {size}")
    X, Y, ids = load_resize(pairs, size)
    np.savez(out, X=X, Y=Y, ids=ids)
    return out


def load_cache(path):
    d = np.load(path, allow_pickle=False)
    return d["X"], d["Y"], d["ids"]


def make_split(n: int, val_frac: float = 0.1, seed: int = 42):
    """Fixed train/validation split of the official training set (same for every run)."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    n_val = int(round(n * val_frac))
    return np.sort(idx[n_val:]), np.sort(idx[:n_val])


# ----------------------------------------------------------------------------
# 2. Augmentation (numpy/cv2 - no extra dependency)
# ----------------------------------------------------------------------------
def augment(img: np.ndarray, msk: np.ndarray, rng: np.random.Generator):
    """img uint8 HxWx3, msk uint8 HxW -> augmented copies.

    Flips, 90-degree rotations, random affine (rotation +-30 deg, scale 0.9-1.1,
    shift +-6%), and colour jitter in HSV (brightness, contrast, saturation, hue).
    """
    h, w = msk.shape
    if rng.random() < 0.5:
        img, msk = img[:, ::-1], msk[:, ::-1]
    if rng.random() < 0.5:
        img, msk = img[::-1], msk[::-1]
    k = int(rng.integers(4))
    if k:
        img, msk = np.rot90(img, k), np.rot90(msk, k)
    img, msk = np.ascontiguousarray(img), np.ascontiguousarray(msk)
    if rng.random() < 0.7:
        ang = rng.uniform(-30, 30)
        sc = rng.uniform(0.9, 1.1)
        M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, sc)
        M[:, 2] += rng.uniform(-0.0625, 0.0625, 2) * (w, h)
        img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
        msk = cv2.warpAffine(msk, M, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_REFLECT_101)
    if rng.random() < 0.7:
        f = img.astype(np.float32)
        f = (f - f.mean()) * rng.uniform(0.8, 1.2) + f.mean() * rng.uniform(0.8, 1.2)
        f = np.clip(f, 0, 255).astype(np.uint8)
        hsv = cv2.cvtColor(f, cv2.COLOR_RGB2HSV).astype(np.int16)
        hsv[..., 0] = (hsv[..., 0] + int(rng.integers(-5, 6))) % 180
        hsv[..., 1] = np.clip(hsv[..., 1] * rng.uniform(0.8, 1.2), 0, 255)
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    return img, msk


def normalise(img_uint8: np.ndarray) -> np.ndarray:
    """uint8 HxWx3 -> float32 3xHxW, ImageNet-normalised."""
    f = img_uint8.astype(np.float32) / 255.0
    f = (f - IMAGENET_MEAN) / IMAGENET_STD
    return np.transpose(f, (2, 0, 1)).copy()


# ----------------------------------------------------------------------------
# 3. Metrics (per image)
# ----------------------------------------------------------------------------
def _surface(mask: np.ndarray) -> np.ndarray:
    return mask & ~ndimage.binary_erosion(mask, structure=np.ones((3, 3)), border_value=0)


def hd95(pred: np.ndarray, gt: np.ndarray) -> float:
    """95th-percentile symmetric Hausdorff distance in pixels. NaN when either mask is empty."""
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() or not gt.any():
        return float("nan")
    sp, sg = _surface(pred), _surface(gt)
    dt_g = ndimage.distance_transform_edt(~sg)
    dt_p = ndimage.distance_transform_edt(~sp)
    d = np.concatenate([dt_g[sp], dt_p[sg]])
    return float(np.percentile(d, 95))


def image_metrics(pred: np.ndarray, gt: np.ndarray, with_hd: bool = True) -> dict:
    """pred, gt: binary HxW. Empty-vs-empty counts as perfect overlap."""
    p, g = pred.astype(bool), gt.astype(bool)
    tp = float(np.logical_and(p, g).sum())
    fp = float(np.logical_and(p, ~g).sum())
    fn = float(np.logical_and(~p, g).sum())
    tn = float(p.size - tp - fp - fn)
    both_empty = (tp + fp + fn) == 0
    dice = 1.0 if both_empty else 2 * tp / (2 * tp + fp + fn)
    jac = 1.0 if both_empty else tp / (tp + fp + fn)
    prec = (1.0 if both_empty else 0.0) if (tp + fp) == 0 else tp / (tp + fp)
    rec = 1.0 if (tp + fn) == 0 else tp / (tp + fn)
    spec = 1.0 if (tn + fp) == 0 else tn / (tn + fp)
    out = {
        "accuracy": (tp + tn) / p.size,
        "dice": dice,
        "jaccard": jac,
        "thresholded_jaccard": jac if jac >= 0.65 else 0.0,
        "precision": prec,
        "recall": rec,
        "specificity": spec,
    }
    if with_hd:
        out["hd95"] = hd95(p, g)
    return out


METRICS = ["accuracy", "dice", "jaccard", "thresholded_jaccard", "precision", "recall", "specificity", "hd95"]
HIGHER_IS_BETTER = {m: (m != "hd95") for m in METRICS}


def evaluate_masks(preds: np.ndarray, gts: np.ndarray, ids, with_hd: bool = True) -> pd.DataFrame:
    rows = []
    for i in range(len(preds)):
        r = image_metrics(preds[i], gts[i], with_hd)
        r["image_id"] = str(ids[i])
        rows.append(r)
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# 4. Results bookkeeping and statistics
# ----------------------------------------------------------------------------
def run_dir(results_dir, experiment, model, seed) -> Path:
    return Path(results_dir) / experiment / f"{model}__seed{seed}"


def is_done(results_dir, experiment, model, seed) -> bool:
    return (run_dir(results_dir, experiment, model, seed) / "done.json").exists()


def collect_per_image(results_dir, experiment, split) -> pd.DataFrame:
    """Stack every run's per-image metrics for one evaluation split."""
    frames = []
    for d in sorted(Path(results_dir, experiment).glob("*__seed*")):
        f = d / f"metrics_{split}.csv"
        if not (d / "done.json").exists() or not f.exists():
            continue
        model, seed = d.name.split("__seed")
        df = pd.read_csv(f)
        df["model"], df["seed"] = model, int(seed)
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def summarise(per_image: pd.DataFrame, metrics=METRICS, order=None) -> pd.DataFrame:
    """Mean over images per run, then mean, SD and 95% t-CI across seeds per model."""
    metrics = [m for m in metrics if m in per_image.columns]
    run = per_image.groupby(["model", "seed"])[metrics].mean().reset_index()
    rows = []
    for model, g in run.groupby("model"):
        n = len(g)
        row = {"model": model, "n_seeds": n}
        for m in metrics:
            v = g[m].to_numpy(float)
            mu = float(np.nanmean(v))
            sd = float(np.nanstd(v, ddof=1)) if n > 1 else float("nan")
            half = float(stats.t.ppf(0.975, n - 1) * sd / math.sqrt(n)) if n > 1 else float("nan")
            row[m], row[f"{m}_sd"], row[f"{m}_ci95"] = mu, sd, half
        rows.append(row)
    out = pd.DataFrame(rows)
    if order:
        out["__o"] = out["model"].map({m: i for i, m in enumerate(order)})
        out = out.sort_values("__o").drop(columns="__o")
    return out.reset_index(drop=True)


def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    m = len(p)
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


def paired_tests(per_image: pd.DataFrame, reference: str, metric: str = "jaccard") -> pd.DataFrame:
    """Wilcoxon signed-rank test of each model vs `reference` on per-image `metric`
    (averaged over seeds first, so each image is one paired observation), Holm-corrected.
    Also reports a Friedman test across all models and the matched-pairs rank-biserial effect size."""
    piv = per_image.groupby(["image_id", "model"])[metric].mean().unstack("model").dropna()
    if reference not in piv.columns:
        raise KeyError(f"{reference} not in results")
    rows = []
    for m in piv.columns:
        if m == reference:
            continue
        diff = piv[m] - piv[reference]
        nz = diff[diff != 0]
        if len(nz) == 0:
            stat, p, rbc = float("nan"), 1.0, 0.0
        else:
            stat, p = stats.wilcoxon(piv[m], piv[reference], zero_method="wilcox")
            ranks = stats.rankdata(np.abs(nz))
            rbc = float((ranks[nz > 0].sum() - ranks[nz < 0].sum()) / ranks.sum())
        rows.append({"model": m, "vs": reference, "metric": metric, "n_images": len(piv),
                     "mean_diff": float(diff.mean()), "median_diff": float(diff.median()),
                     "wilcoxon_W": float(stat), "p_value": float(p), "rank_biserial": rbc})
    out = pd.DataFrame(rows)
    if len(out):
        out["p_holm"] = holm(out["p_value"])
        out["significant_0.05"] = out["p_holm"] < 0.05
    if piv.shape[1] >= 3:
        chi, pf = stats.friedmanchisquare(*[piv[c] for c in piv.columns])
        out.attrs["friedman"] = {"chi2": float(chi), "p": float(pf), "k": piv.shape[1], "n": len(piv)}
    return out


def fmt_table(summary: pd.DataFrame, metrics, labels=None, decimals=4, bold_best=True, latex=False) -> str:
    """Markdown (or LaTeX) table of mean +- SD, best per column in bold."""
    labels = labels or {}
    metrics = [m for m in metrics if m in summary.columns]
    best = {}
    for m in metrics:
        col = summary[m]
        best[m] = col.idxmax() if HIGHER_IS_BETTER.get(m, True) else col.idxmin()
    names = {"accuracy": "Accuracy", "dice": "Dice (F1)", "jaccard": "Jaccard (IoU)",
             "thresholded_jaccard": "Thresholded Jaccard", "precision": "Precision",
             "recall": "Recall", "specificity": "Specificity", "hd95": "HD95 (px)"}
    header = ["Model"] + [names.get(m, m) for m in metrics]
    lines = []
    for i, r in summary.iterrows():
        cells = [labels.get(r["model"], r["model"])]
        for m in metrics:
            d = 2 if m == "hd95" else decimals
            sd = r.get(f"{m}_sd", float("nan"))
            s = f"{r[m]:.{d}f}" + ("" if pd.isna(sd) else f" ± {sd:.{d}f}")
            if bold_best and i == best[m]:
                s = (r"\textbf{" + s + "}") if latex else f"**{s}**"
            cells.append(s.replace("±", r"$\pm$") if latex else s)
        lines.append(cells)
    if latex:
        out = ["\\begin{tabular}{l" + "c" * len(metrics) + "}", "\\toprule", " & ".join(header) + r" \\", "\\midrule"]
        out += [" & ".join(c) + r" \\" for c in lines]
        out += ["\\bottomrule", "\\end{tabular}"]
        return "\n".join(out)
    out = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    out += ["| " + " | ".join(c) + " |" for c in lines]
    return "\n".join(out)


# ----------------------------------------------------------------------------
# 5. Figures (matplotlib, 300 dpi, colour-blind-checked palette)
# ----------------------------------------------------------------------------
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]


def _plt():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                         "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.6,
                         "axes.edgecolor": "#8a8984", "savefig.dpi": 300, "savefig.bbox": "tight"})
    return plt


def fig_box(per_image: pd.DataFrame, models, labels, metric, out_path, title=None):
    """Per-image distribution (seeds averaged) of one metric, one box per model."""
    plt = _plt()
    piv = per_image.groupby(["image_id", "model"])[metric].mean().unstack("model")
    models = [m for m in models if m in piv.columns]
    data = [piv[m].dropna().to_numpy() for m in models]
    fig, ax = plt.subplots(figsize=(max(5, 1.1 * len(models) + 1.5), 3.6))
    bp = ax.boxplot(data, widths=0.55, patch_artist=True, showfliers=False, medianprops={"color": "#0b0b0b"})
    for i, b in enumerate(bp["boxes"]):
        b.set_facecolor(PALETTE[i % 8]); b.set_alpha(0.35); b.set_edgecolor(PALETTE[i % 8])
    for i, d in enumerate(data):
        ax.scatter(np.full(len(d), i + 1) + np.random.default_rng(0).uniform(-0.18, 0.18, len(d)), d,
                   s=3, color=PALETTE[i % 8], alpha=0.25, linewidths=0)
        ax.scatter([i + 1], [d.mean()], marker="D", s=22, color="#0b0b0b", zorder=3)
    ax.set_xticks(range(1, len(models) + 1), [labels.get(m, m) for m in models], rotation=20, ha="right")
    ax.set_ylabel({"jaccard": "Jaccard (IoU) per image", "dice": "Dice per image", "hd95": "HD95 (px) per image"}.get(metric, metric))
    if title:
        ax.set_title(title, loc="left", fontsize=11)
    fig.savefig(out_path); plt.close(fig)


def fig_curves(results_dir, experiment, models, labels, seed, out_path, key="val_dice"):
    """Validation curve of one seed per model on a shared axis."""
    plt = _plt()
    fig, ax = plt.subplots(figsize=(6, 3.6))
    for i, m in enumerate(models):
        f = run_dir(results_dir, experiment, m, seed) / "history.csv"
        if not f.exists():
            continue
        h = pd.read_csv(f)
        ax.plot(h["epoch"], h[key], lw=1.8, color=PALETTE[i % 8], label=labels.get(m, m))
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Epoch"); ax.set_ylabel({"val_dice": "Validation Dice", "val_loss": "Validation loss"}.get(key, key))
    ax.legend(frameon=False, fontsize=8, ncol=2)
    fig.savefig(out_path); plt.close(fig)


def fig_train_val(history_csv, out_path, title=None):
    """Training vs validation loss and Dice for one run (two panels, shared epochs)."""
    plt = _plt()
    h = pd.read_csv(history_csv)
    fig, axs = plt.subplots(1, 2, figsize=(8, 3.2))
    for ax, (a, b, lab) in zip(axs, [("train_loss", "val_loss", "Loss"), ("train_dice", "val_dice", "Dice")]):
        ax.plot(h["epoch"], h[a], color=PALETTE[0], lw=1.8, label="Training")
        ax.plot(h["epoch"], h[b], color=PALETTE[1], lw=1.8, label="Validation")
        ax.set_xlabel("Epoch"); ax.set_ylabel(lab)
        from matplotlib.ticker import MaxNLocator
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    axs[0].legend(frameon=False)
    if title:
        fig.suptitle(title, x=0.01, ha="left", fontsize=11)
    fig.savefig(out_path); plt.close(fig)


def fig_qualitative(X, Y, ids, preds_by_model: dict, labels, pick_ids, out_path):
    """Rows = cases, columns = image + GT + each model. Contours: GT white, prediction in model colour."""
    plt = _plt()
    models = list(preds_by_model)
    idx = [int(np.where(ids == p)[0][0]) for p in pick_ids]
    ncol = 2 + len(models)
    fig, axs = plt.subplots(len(idx), ncol, figsize=(1.6 * ncol, 1.6 * len(idx)))
    axs = np.atleast_2d(axs)
    for r, i in enumerate(idx):
        img = X[i]
        axs[r, 0].imshow(img)
        axs[r, 1].imshow(Y[i], cmap="gray")
        for c, m in enumerate(models):
            ax = axs[r, 2 + c]
            ax.imshow(img)
            ax.contour(Y[i], levels=[0.5], colors="white", linewidths=1.0)
            ax.contour(preds_by_model[m][i], levels=[0.5], colors=PALETTE[c % 8], linewidths=1.2)
            j = image_metrics(preds_by_model[m][i], Y[i], with_hd=False)["jaccard"]
            ax.text(4, 14, f"J={j:.2f}", color="white", fontsize=7, bbox=dict(fc="black", alpha=0.5, lw=0, pad=1))
        for ax in axs[r]:
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for c, t in enumerate(["Image", "Ground truth"] + [labels.get(m, m) for m in models]):
        axs[0, c].set_title(t, fontsize=8)
    fig.savefig(out_path); plt.close(fig)

`isicseg_train.py`: written to the project folder so the analysis can be re-run later without the notebook.

In [ ]:
%%writefile {CODE}/isicseg_train.py
"""
isicseg_train.py - models, training loop (resumable), inference with TTA.

Requires torch and segmentation-models-pytorch >= 0.4 (for SegFormer).
"""
from __future__ import annotations

import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import isicseg_core as core

# ----------------------------------------------------------------------------
# Models
# ----------------------------------------------------------------------------
class DoubleConv(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class AttentionGate(nn.Module):
    """Additive attention gate of Oktay et al. (2018)."""
    def __init__(self, c_skip, c_gate, c_int):
        super().__init__()
        self.wx = nn.Sequential(nn.Conv2d(c_skip, c_int, 1, bias=False), nn.BatchNorm2d(c_int))
        self.wg = nn.Sequential(nn.Conv2d(c_gate, c_int, 1, bias=False), nn.BatchNorm2d(c_int))
        self.psi = nn.Sequential(nn.Conv2d(c_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())

    def forward(self, x, g):
        a = F.relu(self.wx(x) + self.wg(g), inplace=True)
        return x * self.psi(a)


class UNetScratch(nn.Module):
    """Classic U-Net (Ronneberger et al., 2015) with BatchNorm; optional attention gates."""
    def __init__(self, base=64, attention=False):
        super().__init__()
        c = [base, base * 2, base * 4, base * 8, base * 16]
        self.enc = nn.ModuleList([DoubleConv(3, c[0])] + [DoubleConv(c[i], c[i + 1]) for i in range(4)])
        self.up = nn.ModuleList([nn.ConvTranspose2d(c[i + 1], c[i], 2, stride=2) for i in reversed(range(4))])
        self.dec = nn.ModuleList([DoubleConv(c[i] * 2, c[i]) for i in reversed(range(4))])
        self.att = nn.ModuleList([AttentionGate(c[i], c[i], max(c[i] // 2, 1)) for i in reversed(range(4))]) if attention else None
        self.head = nn.Conv2d(c[0], 1, 1)

    def forward(self, x):
        skips = []
        for i, blk in enumerate(self.enc):
            x = blk(x if i == 0 else F.max_pool2d(x, 2))
            skips.append(x)
        x = skips.pop()
        for j, (up, dec) in enumerate(zip(self.up, self.dec)):
            x = up(x)
            s = skips.pop()
            if self.att is not None:
                s = self.att[j](s, x)
            x = dec(torch.cat([s, x], 1))
        return self.head(x)


class SizeGuard(nn.Module):
    """Makes sure logits come back at input resolution (some decoders output 1/4)."""
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, x):
        y = self.net(x)
        if y.shape[-2:] != x.shape[-2:]:
            y = F.interpolate(y, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return y


MODEL_LABELS = core.MODEL_LABELS


def build_model(name: str, pretrained: bool = True) -> nn.Module:
    w = "imagenet" if pretrained else None
    if name == "unet_scratch":
        net = UNetScratch(64, attention=False)
    elif name == "attunet_scratch":
        net = UNetScratch(64, attention=True)
    else:
        import segmentation_models_pytorch as smp
        if name == "unet_r34":
            net = smp.Unet("resnet34", encoder_weights=w, in_channels=3, classes=1)
        elif name == "unetpp_r34":
            net = smp.UnetPlusPlus("resnet34", encoder_weights=w, in_channels=3, classes=1)
        elif name == "deeplabv3p_r34":
            net = smp.DeepLabV3Plus("resnet34", encoder_weights=w, in_channels=3, classes=1)
        elif name == "segformer_b2":
            net = smp.Segformer("mit_b2", encoder_weights=w, in_channels=3, classes=1)
        else:
            raise ValueError(f"unknown model {name}")
    return SizeGuard(net)


def count_params(model) -> int:
    return sum(p.numel() for p in model.parameters())


# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
class SegDataset(torch.utils.data.Dataset):
    def __init__(self, X, Y, idx, train: bool):
        self.X, self.Y, self.idx, self.train = X, Y, np.asarray(idx), train
        self.rng = None

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, k):
        if self.train and self.rng is None:
            info = torch.utils.data.get_worker_info()
            self.rng = np.random.default_rng((info.seed if info else torch.initial_seed()) % (2 ** 32))
        i = self.idx[k]
        img, msk = self.X[i], self.Y[i]
        if self.train:
            img, msk = core.augment(img, msk, self.rng)
        return torch.from_numpy(core.normalise(img)), torch.from_numpy(msk.astype(np.float32))[None]


# ----------------------------------------------------------------------------
# Loss
# ----------------------------------------------------------------------------
def soft_dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits.float())
    t = target.float()
    inter = (p * t).flatten(1).sum(1)
    den = p.flatten(1).sum(1) + t.flatten(1).sum(1)
    return 1 - ((2 * inter + eps) / (den + eps)).mean()


def make_loss(kind: str):
    bce = nn.BCEWithLogitsLoss()
    if kind == "bce":
        return lambda l, t: bce(l.float(), t)
    if kind == "dice":
        return soft_dice_loss
    if kind == "bce_dice":
        return lambda l, t: bce(l.float(), t) + soft_dice_loss(l, t)
    raise ValueError(kind)


# ----------------------------------------------------------------------------
# Inference
# ----------------------------------------------------------------------------
@torch.no_grad()
def predict_probs(model, X, device, batch=32, tta=False, idx=None) -> np.ndarray:
    """Returns uint8 probabilities (0-255), shape N x H x W. TTA = mean of identity, h-flip, v-flip."""
    model.eval()
    idx = np.arange(len(X)) if idx is None else np.asarray(idx)
    out = np.zeros((len(idx),) + X.shape[1:3], np.uint8)
    use_amp = device.type == "cuda"
    for s in range(0, len(idx), batch):
        xb = torch.from_numpy(np.stack([core.normalise(X[i]) for i in idx[s:s + batch]])).to(device)
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
            p = torch.sigmoid(model(xb).float())
            if tta:
                p = p + torch.sigmoid(model(xb.flip(-1)).float()).flip(-1)
                p = p + torch.sigmoid(model(xb.flip(-2)).float()).flip(-2)
                p = p / 3
        out[s:s + len(xb)] = (p[:, 0].float().cpu().numpy() * 255).round().astype(np.uint8)
    return out


def hard_dice(probs_u8, gts):
    p = probs_u8 >= 128
    g = gts.astype(bool)
    inter = (p & g).reshape(len(p), -1).sum(1)
    den = p.reshape(len(p), -1).sum(1) + g.reshape(len(g), -1).sum(1)
    d = np.where(den == 0, 1.0, 2 * inter / np.maximum(den, 1))
    return float(d.mean())


# ----------------------------------------------------------------------------
# Training (one run = one model x one seed x one config), resumable per epoch
# ----------------------------------------------------------------------------
DEFAULT_CFG = dict(
    max_epochs=100, batch_size=16, lr=3e-4, weight_decay=1e-4, optimizer="adamw", scheduler="cosine",
    loss="bce_dice", augment=True, pretrained=True, monitor="val_dice", patience=15, restore_best=True,
    early_stopping=True, tta=True, num_workers=2, save_weights=False, save_pred=False, ckpt_every=5,
)


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def train_run(model_name, seed, cfg, data, out_dir, eval_sets, device=None, log_every=1):
    """
    data: dict with X, Y (training pool arrays) and train_idx, val_idx.
    eval_sets: dict name -> (X, Y, ids) evaluated after training.
    Writes history.csv, metrics_<set>.csv, preds_<set>.npz (optional), done.json.
    """
    cfg = {**DEFAULT_CFG, **cfg}
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    if (out / "done.json").exists():
        print(f"[skip] {out.name} already done"); return json.loads((out / "done.json").read_text())
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_seed(seed)
    X, Y = data["X"], data["Y"]
    model = build_model(model_name, cfg["pretrained"]).to(device)
    if cfg["optimizer"] == "adamw":
        opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    else:
        opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["max_epochs"], eta_min=cfg["lr"] * 0.01)
             if cfg["scheduler"] == "cosine" else None)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    loss_fn = make_loss(cfg["loss"])
    g = torch.Generator(); g.manual_seed(seed)
    dl = torch.utils.data.DataLoader(SegDataset(X, Y, data["train_idx"], cfg["augment"]), batch_size=cfg["batch_size"],
                                     shuffle=True, drop_last=True, num_workers=cfg["num_workers"],
                                     pin_memory=use_amp, generator=g)
    vX, vY = X[data["val_idx"]], Y[data["val_idx"]]

    start, hist, best_score, best_state, best_epoch, wait = 1, [], None, None, 0, 0
    ck = out / "last.pt"
    s = None
    if ck.exists():  # resume after a disconnect
        try:
            s = torch.load(ck, map_location="cpu", weights_only=False)
        except Exception as e:  # checkpoint cut off by a disconnect: start this run again
            print(f"[warn] {ck} is unreadable ({type(e).__name__}); restarting {out.name} from epoch 1")
            ck.unlink(missing_ok=True)
            s = None
    if s is not None:
        model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"]); scaler.load_state_dict(s["scaler"])
        if sched and s["sched"]:
            sched.load_state_dict(s["sched"])
        start, hist, best_score, best_epoch, wait = s["epoch"] + 1, s["hist"], s["best_score"], s["best_epoch"], s["wait"]
        best_state = s["best_state"]
        torch.set_rng_state(s["rng_cpu"]); g.set_state(s["rng_gen"])
        print(f"[resume] {out.name} from epoch {start}")

    maximise = cfg["monitor"] != "val_loss"
    for epoch in range(start, cfg["max_epochs"] + 1):
        model.train()
        ep_t0 = time.time()
        tl, td, nb = 0.0, 0.0, 0
        for xb, yb in dl:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(xb)
            loss = loss_fn(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            with torch.no_grad():
                td += 1 - soft_dice_loss((logits.float() > 0).float() * 40 - 20, yb).item()
            tl += loss.item(); nb += 1
        if sched:
            sched.step()
        # validation
        model.eval()
        vl, vn = 0.0, 0
        with torch.no_grad():
            for s in range(0, len(vX), 32):
                xb = torch.from_numpy(np.stack([core.normalise(im) for im in vX[s:s + 32]])).to(device)
                yb = torch.from_numpy(vY[s:s + 32].astype(np.float32))[:, None].to(device)
                with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                    lg = model(xb)
                vl += loss_fn(lg, yb).item() * len(xb); vn += len(xb)
        vprob = predict_probs(model, vX, device, tta=False)
        row = {"epoch": epoch, "train_loss": tl / max(nb, 1), "train_dice": td / max(nb, 1),
               "val_loss": vl / vn, "val_dice": hard_dice(vprob, vY), "lr": opt.param_groups[0]["lr"],
               "seconds": time.time() - ep_t0}
        hist.append(row)
        score = row[cfg["monitor"]]
        improved = best_score is None or (score > best_score if maximise else score < best_score)
        if improved:
            best_score, best_epoch, wait = score, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        if epoch % log_every == 0:
            print(f"{out.name} ep {epoch:3d} | train {row['train_loss']:.4f}/{row['train_dice']:.4f} "
                  f"| val {row['val_loss']:.4f}/{row['val_dice']:.4f} | best {best_epoch}")
        pd.DataFrame(hist).to_csv(out / "history.csv", index=False)
        stopping = (cfg["early_stopping"] and wait >= cfg["patience"]) or bool(cfg.get("_stop_after") and epoch >= cfg["_stop_after"])
        if epoch % cfg["ckpt_every"] == 0 or stopping:
          torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                      "sched": sched.state_dict() if sched else None, "epoch": epoch, "hist": hist,
                      "best_score": best_score, "best_epoch": best_epoch, "wait": wait, "best_state": best_state,
                      "rng_cpu": torch.get_rng_state(), "rng_gen": g.get_state()}, ck.with_suffix(".tmp"))
          os.replace(ck.with_suffix(".tmp"), ck)  # atomic: a disconnect never leaves a half-written last.pt
        if cfg.get("_stop_after") and epoch >= cfg["_stop_after"]:  # used by the smoke test to simulate a disconnect
            print(f"{out.name}: stopping session after epoch {epoch} (resume test)")
            return None
        if cfg["early_stopping"] and wait >= cfg["patience"]:
            print(f"{out.name}: early stop at epoch {epoch} (best {best_epoch})")
            break
    epochs_run = hist[-1]["epoch"]
    if cfg["restore_best"] and best_state is not None:
        model.load_state_dict(best_state)
    train_seconds = float(sum(h["seconds"] for h in hist))

    # evaluation
    summary = {"model": model_name, "seed": seed, "cfg": cfg, "epochs_run": epochs_run, "best_epoch": best_epoch,
               "best_val": best_score, "params": count_params(model), "train_seconds": train_seconds}
    for name, (eX, eY, eids) in eval_sets.items():
        t1 = time.time()
        probs = predict_probs(model, eX, device, tta=cfg["tta"])
        dt = time.time() - t1
        preds = (probs >= 128).astype(np.uint8)
        df = core.evaluate_masks(preds, eY, eids)
        df.to_csv(out / f"metrics_{name}.csv", index=False)
        summary[f"{name}_ms_per_image"] = 1000 * dt / len(eX)
        summary[name] = {m: float(df[m].mean()) for m in core.METRICS}
        if cfg["save_pred"]:
            np.savez_compressed(out / f"preds_{name}.npz", probs=probs, ids=eids)
        print(f"  {name}: Dice {summary[name]['dice']:.4f}  Jaccard {summary[name]['jaccard']:.4f}")
    if cfg["save_weights"]:
        torch.save(model.state_dict(), out / "best_weights.pt")
    (out / "done.json").write_text(json.dumps(summary, indent=2, default=str))
    ck.unlink(missing_ok=True)
    return summary

`isicseg_experiments.py`: written to the project folder so the analysis can be re-run later without the notebook.

In [ ]:
%%writefile {CODE}/isicseg_experiments.py
"""
isicseg_experiments.py - experiment definitions, runners and the analysis stage.

The analysis functions need only numpy/pandas/scipy/matplotlib; the runners
import torch lazily, so analysis can be re-run anywhere from the results folder.
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

import isicseg_core as core

# ----------------------------------------------------------------------------
# Experiment definitions
# ----------------------------------------------------------------------------
BENCHMARK_MODELS = ["unet_scratch", "attunet_scratch", "unet_r34", "unetpp_r34", "deeplabv3p_r34", "segformer_b2"]
SEEDS = [0, 1, 2, 3, 4]

# Experiment A: every model, same modern recipe (AdamW, cosine LR, BCE+Dice, augmentation,
# early stopping on validation Dice with best-weight restoration, flip TTA at test time).
BENCHMARK_CFG = dict(max_epochs=100, batch_size=16, lr=3e-4, weight_decay=1e-4, optimizer="adamw",
                     scheduler="cosine", loss="bce_dice", augment=True, pretrained=True,
                     monitor="val_dice", patience=15, restore_best=True, early_stopping=True, tta=True)

# Experiment B: replication of the four training protocols of the original study on the
# scratch U-Net (Adam 1e-4, constant LR, BCE, no augmentation, early stopping on val loss).
_PROTO_BASE = dict(optimizer="adam", lr=1e-4, scheduler="none", loss="bce", augment=False, pretrained=False,
                   monitor="val_loss", tta=False)
PROTOCOLS = {
    "T1": {**_PROTO_BASE, "max_epochs": 5, "batch_size": 4, "early_stopping": False, "restore_best": False, "patience": 0},
    "T2": {**_PROTO_BASE, "max_epochs": 100, "batch_size": 8, "early_stopping": True, "patience": 20, "restore_best": False},
    "T3": {**_PROTO_BASE, "max_epochs": 25, "batch_size": 8, "early_stopping": True, "patience": 10, "restore_best": True},
    "T4": {**_PROTO_BASE, "max_epochs": 100, "batch_size": 16, "early_stopping": True, "patience": 10, "restore_best": True},
}
PROTOCOL_LABELS = {"T1": "T1: 5 ep, bs 4", "T2": "T2: bs 8, P20, last weights",
                   "T3": "T3: 25 ep, bs 8, P10, best weights", "T4": "T4: bs 16, P10, best weights"}

TABLE_METRICS = ["accuracy", "dice", "jaccard", "thresholded_jaccard", "precision", "recall", "hd95"]


# ----------------------------------------------------------------------------
# Data preparation
# ----------------------------------------------------------------------------
def prepare_data(raw_dir, cache_dir, size=256, external=("isic2017_test",), ph2_root=None, val_frac=0.1):
    """Builds/loads caches. Returns (train data dict, eval_sets dict)."""
    paths = {n: core.build_cache(n, raw_dir, cache_dir, size) for n in ("isic2018_train", "isic2018_val", "isic2018_test")}
    for n in external:
        paths[n] = core.build_cache(n, raw_dir, cache_dir, size)
    if ph2_root:
        p = core.build_cache("ph2", raw_dir, cache_dir, size, ph2_root=ph2_root)
        if p:
            paths["ph2"] = p
    X, Y, ids = core.load_cache(paths["isic2018_train"])
    vX, vY, vids = core.load_cache(paths["isic2018_val"])
    # pool = official training (2,594) + official validation (100); fixed 90/10 split for early stopping
    PX, PY, Pids = np.concatenate([X, vX]), np.concatenate([Y, vY]), np.concatenate([ids, vids])
    tr, va = core.make_split(len(PX), val_frac=val_frac, seed=42)
    data = {"X": PX, "Y": PY, "ids": Pids, "train_idx": tr, "val_idx": va}
    eval_sets = {"isic2018_test": core.load_cache(paths["isic2018_test"])}
    for n in list(external) + (["ph2"] if "ph2" in paths else []):
        eval_sets[n] = core.load_cache(paths[n])
    print(f"train {len(tr)} | val {len(va)} | " + " | ".join(f"{k} {len(v[0])}" for k, v in eval_sets.items()))
    return data, eval_sets


# ----------------------------------------------------------------------------
# Runners (resumable: finished runs are skipped, unfinished runs resume from last epoch)
# ----------------------------------------------------------------------------
def run_benchmark(data, eval_sets, results_dir, models=BENCHMARK_MODELS, seeds=SEEDS, cfg_overrides=None, pred_seed=0):
    import isicseg_train as tr
    out = []
    for seed in seeds:  # seed-major order: one full set of models first, so partial results are usable
        for m in models:
            cfg = {**BENCHMARK_CFG, **(cfg_overrides or {}), "save_pred": seed == pred_seed}
            d = core.run_dir(results_dir, "benchmark", m, seed)
            out.append(tr.train_run(m, seed, cfg, data, d, eval_sets))
    return out


def run_protocols(data, eval_sets, results_dir, protocols=tuple(PROTOCOLS), seeds=SEEDS, pred_seed=0):
    import isicseg_train as tr
    sets = {"isic2018_test": eval_sets["isic2018_test"]}
    out = []
    for seed in seeds:
        for p in protocols:
            cfg = {**PROTOCOLS[p], "save_pred": seed == pred_seed}
            d = core.run_dir(results_dir, "protocol", p, seed)
            out.append(tr.train_run("unet_scratch", seed, cfg, data, d, sets))
    return out


# ----------------------------------------------------------------------------
# Analysis
# ----------------------------------------------------------------------------
def _done(results_dir, experiment):
    rows = []
    for f in sorted(Path(results_dir, experiment).glob("*__seed*/done.json")):
        s = json.loads(f.read_text())
        s["run"] = f.parent.name.split("__seed")[0]
        rows.append(s)
    return rows


def efficiency_table(results_dir, experiment, labels):
    rows = []
    for s in _done(results_dir, experiment):
        rows.append({"model": s["run"], "params_M": s["params"] / 1e6, "epochs_run": s["epochs_run"],
                     "best_epoch": s["best_epoch"], "train_minutes": s.get("train_seconds", np.nan) / 60,
                     "ms_per_image_test": s.get("isic2018_test_ms_per_image", np.nan)})
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows).groupby("model").agg(["mean", "std"]).round(3)
    df.columns = [f"{a}_{b}" for a, b in df.columns]
    df.insert(0, "label", [labels.get(m, m) for m in df.index])
    return df.reset_index()


def analyse(results_dir, out_dir, data=None, eval_sets=None):
    """Writes tables (md/tex/csv), statistical tests, figures and summary.md into out_dir."""
    results_dir, out = Path(results_dir), Path(out_dir)
    (out / "tables").mkdir(parents=True, exist_ok=True); (out / "figures").mkdir(exist_ok=True); (out / "stats").mkdir(exist_ok=True)
    md = ["# Results summary\n"]

    for exp, order, labels in [("benchmark", BENCHMARK_MODELS, core.MODEL_LABELS),
                               ("protocol", list(PROTOCOLS), PROTOCOL_LABELS)]:
        if not (results_dir / exp).exists():
            continue
        splits = sorted({f.stem.replace("metrics_", "") for f in (results_dir / exp).glob("*/metrics_*.csv")})
        for split in splits:
            pim = core.collect_per_image(results_dir, exp, split)
            if pim.empty:
                continue
            summ = core.summarise(pim, TABLE_METRICS, order=order)
            tag = f"{exp}_{split}"
            summ.to_csv(out / "tables" / f"{tag}.csv", index=False)
            t_md = core.fmt_table(summ, TABLE_METRICS, labels)
            (out / "tables" / f"{tag}.md").write_text(t_md)
            (out / "tables" / f"{tag}.tex").write_text(core.fmt_table(summ, TABLE_METRICS, labels, latex=True))
            seeds = ", ".join(f"{labels.get(m, m)}: {n}" for m, n in zip(summ["model"], summ["n_seeds"]))
            md += [f"## {exp} - {split}\n", f"Mean ± SD across seeds (seeds per model: {seeds}).\n", t_md, ""]
            md.append("| Model | Jaccard 95% CI |\n|---|---|")
            md += [f"| {labels.get(r.model, r.model)} | {r.jaccard:.4f} ± {r.jaccard_ci95:.4f} |" for r in summ.itertuples()]
            md.append("")
            # statistics: vs best model and vs the first model in the order (baseline)
            present = [m for m in order if m in set(pim["model"])]
            if len(present) >= 2:
                best = summ.loc[summ["jaccard"].idxmax(), "model"]
                for ref in dict.fromkeys([best, present[0]]):
                    for metric in ("jaccard", "dice"):
                        tst = core.paired_tests(pim, ref, metric)
                        tst = tst.assign(_o=tst["model"].map({m: i for i, m in enumerate(order)})).sort_values("_o").drop(columns="_o")
                        tst.to_csv(out / "stats" / f"{tag}_wilcoxon_{metric}_vs_{ref}.csv", index=False)
                        if metric == "jaccard":
                            md.append(f"**Wilcoxon signed-rank vs {labels.get(ref, ref)} (per-image Jaccard, Holm-corrected)**\n")
                            md.append("| Model | Mean diff | p (Holm) | Rank-biserial r |\n|---|---|---|---|")
                            md += [f"| {labels.get(r.model, r.model)} | {r.mean_diff:+.4f} | {r.p_holm:.2e} | {r.rank_biserial:+.3f} |"
                                   for r in tst.itertuples()]
                            fr = tst.attrs.get("friedman")
                            if fr:
                                md.append(f"\nFriedman test across {fr['k']} models, {fr['n']} images: "
                                          f"χ² = {fr['chi2']:.1f}, p = {fr['p']:.2e}.")
                                (out / "stats" / f"{tag}_friedman_jaccard.json").write_text(json.dumps(fr, indent=2))
                            md.append("")
                core.fig_box(pim, order, labels, "jaccard", out / "figures" / f"box_jaccard_{tag}.png")
                if "hd95" in pim:
                    core.fig_box(pim, order, labels, "hd95", out / "figures" / f"box_hd95_{tag}.png")
        eff = efficiency_table(results_dir, exp, labels)
        if len(eff):
            eff.to_csv(out / "tables" / f"{exp}_efficiency.csv", index=False)
            md += [f"## {exp} - training and inference cost\n", eff.to_markdown(index=False), ""]
        # learning curves (seed 0)
        present = [m for m in order if core.run_dir(results_dir, exp, m, 0).joinpath("history.csv").exists()]
        if present:
            core.fig_curves(results_dir, exp, present, labels, 0, out / "figures" / f"val_dice_curves_{exp}.png", "val_dice")
            core.fig_curves(results_dir, exp, present, labels, 0, out / "figures" / f"val_loss_curves_{exp}.png", "val_loss")
            for m in present:
                core.fig_train_val(core.run_dir(results_dir, exp, m, 0) / "history.csv",
                                   out / "figures" / f"train_val_{exp}_{m}.png", labels.get(m, m))

    # qualitative figure from saved seed-0 predictions on the ISIC 2018 test set
    if eval_sets is not None and "isic2018_test" in eval_sets:
        X, Y, ids = eval_sets["isic2018_test"]
        preds = {}
        for m in BENCHMARK_MODELS:
            f = core.run_dir(results_dir, "benchmark", m, 0) / "preds_isic2018_test.npz"
            if f.exists():
                z = np.load(f)
                preds[m] = (z["probs"] >= 128).astype(np.uint8)
        if preds:
            ref = next(iter(preds))
            j = np.array([core.image_metrics(preds[ref][i], Y[i], with_hd=False)["jaccard"] for i in range(len(Y))])
            order_idx = np.argsort(j)
            picks = [order_idx[-1], order_idx[int(len(j) * 0.75)], order_idx[len(j) // 2],
                     order_idx[int(len(j) * 0.1)], order_idx[0]]
            core.fig_qualitative(X, Y, ids, preds, core.MODEL_LABELS, [ids[i] for i in picks],
                                 out / "figures" / "qualitative_isic2018_test.png")
            md.append("Qualitative figure rows: best, 75th, 50th, 10th percentile and worst case by "
                      f"{core.MODEL_LABELS.get(ref, ref)} Jaccard.\n")

    (out / "summary.md").write_text("\n".join(md))
    print(f"Analysis written to {out}")
    return out / "summary.md"

## 1. Download and cache the data
The official ISIC zips (about 11 GB) are downloaded once to local disk and resized to `IMG_SIZE`; the cache (under 1 GB) is kept in the project folder.

In [ ]:
import importlib, isicseg_core as core, isicseg_experiments as ex
importlib.reload(core); importlib.reload(ex)
data, eval_sets = ex.prepare_data(RAW, CACHE, IMG_SIZE, external=tuple(EXTERNAL), ph2_root=PH2_ROOT)

## 2. Smoke test
Trains every model for 2 epochs on 64 images, interrupting after epoch 1 and resuming, so problems surface in about 2 minutes instead of hours later.

In [ ]:
import shutil, numpy as np, isicseg_train as tr
importlib.reload(tr)
if RUN_SMOKE_TEST:
    tmp = "/tmp/smoke"; shutil.rmtree(tmp, ignore_errors=True)
    small = {**data, "train_idx": data["train_idx"][:64], "val_idx": data["val_idx"][:16]}
    sets = {k: tuple(a[:8] for a in v) for k, v in eval_sets.items()}
    for m in MODELS:
        cfg = dict(max_epochs=2, batch_size=8, ckpt_every=1, save_pred=True)
        d = f"{tmp}/{m}"
        assert tr.train_run(m, 0, {**cfg, "_stop_after": 1}, small, d, sets) is None   # simulated disconnect
        s = tr.train_run(m, 0, cfg, small, d, sets)                                    # resume and finish
        assert s["epochs_run"] == 2 and os.path.exists(f"{d}/metrics_isic2018_test.csv")
        print(f"OK {m}: {s['params']/1e6:.1f}M params")
    for p in ex.PROTOCOLS:
        s = tr.train_run("unet_scratch", 0, {**ex.PROTOCOLS[p], "max_epochs": 1}, small, f"{tmp}/proto_{p}", {"isic2018_test": sets["isic2018_test"]})
    shutil.rmtree(tmp, ignore_errors=True)
    print("Smoke test passed.")

## 3. Experiment A: architecture benchmark
Resumable: re-running this cell skips finished runs. Seed-0 predictions are saved for the qualitative figure.

In [ ]:
if RUN_BENCHMARK:
    ex.run_benchmark(data, eval_sets, RESULTS, models=MODELS, seeds=SEEDS)

## 4. Experiment B: replication of protocols T1–T4
The scratch U-Net under the four protocols of the original study (Adam 1e-4, BCE, no augmentation, early stopping on validation loss).

In [ ]:
if RUN_PROTOCOLS:
    ex.run_protocols(data, eval_sets, RESULTS, seeds=SEEDS)

## 5. Analysis: tables, statistics and figures
This cell works on partial results too, so you can run it at any time.

In [ ]:
from IPython.display import Markdown, Image, display
importlib.reload(core); importlib.reload(ex)
summary = ex.analyse(RESULTS, ANALYSIS, data, eval_sets)
display(Markdown(open(summary).read()))
for f in ["box_jaccard_benchmark_isic2018_test.png", "qualitative_isic2018_test.png",
          "val_dice_curves_benchmark.png", "box_jaccard_protocol_isic2018_test.png"]:
    p = f"{ANALYSIS}/figures/{f}"
    if os.path.exists(p):
        display(Image(p, width=900))

## 6. Bundle the results
Creates `results_bundle.zip` with the analysis plus every run's per-image metrics and learning curves (no weights). Send this file back to update the paper.

In [ ]:
import zipfile, glob
bundle = f"{PROJECT}/results_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{ANALYSIS}/**/*", recursive=True):
        if os.path.isfile(f): z.write(f, os.path.relpath(f, PROJECT))
    for pat in ("*/*/metrics_*.csv", "*/*/history.csv", "*/*/done.json"):
        for f in glob.glob(f"{RESULTS}/{pat}"):
            z.write(f, os.path.relpath(f, PROJECT))
print(bundle, round(os.path.getsize(bundle) / 1e6, 1), "MB")
if IN_COLAB:
    from google.colab import files
    files.download(bundle)